In [ ]:
#RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
### Read all the files inside directory or database if we working on RAG
###PDF Loader
from langchain_community.document_loaders import PyPDFLoader

pdf = PyPDFLoader("path")
documents = []
document = pdf.load()
documents.extend(document)

In [ ]:
print(len(documents))

In [36]:
###Text Splitting into chunks

def split_documents(documents,chunk_size=300,chunk_overlap=50):
    """Split documents into smaller chunks for better RAG  Performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample Chunks:")

    return split_docs

In [ ]:
chunks = split_documents(documents)
chunks

In [38]:
# Embeddings and VectorStore DB
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:

    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace Model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embeddings_dim)
        """ 

        if not self.model:
            raise ValueError("Model Not Found")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

    ### initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

In [ ]:
#VectorStore

class VectorStore:
    """Manages document embeddings in a ChromaDB Vector Store"""

    def __init__(self,collection_name: str = "pdf_documents", persist_directory: str = "/vector_store"):
        """ 
        Initialize the vector store

        Args:

            collection_name: Name of the ChromaDB Collection
            persist_directory: Direcoty to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize Chroma DB client and collection"""

        try:
            #create persistent ChromaDB Client
            self.client = chromadb.PersistentClient(path = self.persist_directory)
            
            #Create collection in Chroma DB
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector Store initialized.  collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self,documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents 
        """
        if len(documents) !=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")


        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc ,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare meta data
            metadata = dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            #document content

            documents_text.append(doc.page_content)
            #Embedding
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

In [ ]:
#convert the text into embeddings
texts = [doc.page_content for doc in chunks]


#generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)


#store in vector db

vectorstore.add_documents(chunks,embeddings)

In [47]:
#RAG Retrieval
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """ Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self,query:str,top_k: int =3 , distance_threshold: float =1.8) -> List[Dict[str,Any]]:
        """ 
        Retrive relevant documents for a query

        Args:
            query:The search query
            top_k: Number of top results to return
            distance_threshold: Maximum distance allowed. Lower distance means better match.

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, distance threshold:{distance_threshold}")

        # Generate query embedding

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector 


        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #process results
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):

                    if distance <= distance_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'distance':distance,
                            'rank':i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        

rag_retriever = RAGRetriever(vectorstore,embedding_manager)

In [ ]:
rag_retriever

In [ ]:
retrieved_docs = rag_retriever.retrieve(
    "candidate name email phone resume",
    top_k=3,
    distance_threshold=2.0
)

In [ ]:
rag_retriever.retrieve("What is the candidate's name?")

In [ ]:
rag_retriever.retrieve("PROFESSIONAL SUMMARY")

In [ ]:
###Integration Vector db context pipeline with LLM 

#Initialize the llm
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key="API KEY",
    temperature=0.1,
    max_tokens=1024
)

#Retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    #retriever the context
    results =retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    

    ##Generate the answer using Groq LLM 

    prompt = f""" Use the following context to answer the question concisely.

        Context:
        {context}

        Question: {query}

        Answer:"""
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [ ]:
answer = rag_simple("Where does he work ?",rag_retriever,llm)
print(answer)

In [ ]:
#Enhance RAG PIPELINE like we can use extra features like source , page , metadata 
#Advanced RAG Pipeline here we have streaming , citations , history , summarization 